# M5 Q2:人在廣角畫面裡會不會小到偵測不到

**這是全景鏡頭方案的擋路問題。** 第五輪模擬已證明:一台看得到整個廚房的鏡頭,
可以把身份碎裂率從 **18.8% 壓到 0.1%**(見 `docs/M5_模擬預先登記_全景鏡頭_20260825.md`)。

但那個結論有前提:**那台鏡頭要偵測得到人**。鏡頭掛得越高、視野越廣,
畫面裡的人就越小。本筆記本回答:**小到什麼程度就不行了。**

### 為什麼不能沿用既有的小物件實驗

`docs/M3_小物件模擬結果.md` 已經做過同樣的受控實驗,結論是刀縮到 1/4 尺寸時
AP 從 0.209 崩到 0.024。但**那份實驗用的 Roboflow 資料集裡完全沒有「人」這一類**
(14 類全是餐廚器物)。所以它只證明了刀會崩,**沒有回答人會不會**。

而全景鏡頭只需要偵測「人」—— 刀、砧板那些細節事件留給近距鏡頭。

### 判準(預先登記 §4,跑之前已定死)

M5 只需要「人被偵測到」,不需要精確的框。所以**主指標是人的 Recall**:

| Recall | 判定 |
|---|---|
| ≥ 0.90 | 該尺寸可用 |
| 0.75~0.90 | 勉強,會增加 M4 斷軌 → 回頭餵進模擬的 `master_fragment_rate` |
| < 0.75 | 該尺寸不可用 |

### 事先寫下的預測

4. **人比刀耐縮得多**:s=0.25 時刀 AP 只剩 11%,但人的 Recall 預期仍 > 0.75
5. **提高輸入解析度對人的幫助小於對刀** —— 人本來就沒小到接近偵測極限

### 要準備的兩個檔案

| 檔案 | 本機位置 | 大小 |
|---|---|---|
| `test_for_q2.zip` | `data/m3_finetune_r2/test_for_q2.zip` | 17 MB |
| `checkpoint_best_regular (4).pth` | 專案根目錄 | 123 MB |

**用 Colab 左側的「檔案」面板(📁 圖示)直接拖進去**,不要用 `files.upload()` 的小視窗
——那個重複執行時常常不出現。拖曳支援多檔與大檔,等左下角進度跑完再執行 cell 2。

⚠ 含 EPFL(CC-NC)→ 驗證用、不出貨。縮放模擬是**尺寸效應的受控實驗**,
不等於真實廣角鏡頭(還有徑向畸變、俯視角度差異)。

In [ ]:
# 1) 安裝
!nvidia-smi -L
!pip -q install "rfdetr[train]" supervision

In [ ]:
# 2) 偵測已上傳的檔案(本 cell 不會跳出上傳視窗)
#
# ── 怎麼把檔案弄進來 ────────────────────────────────────────────────
# 用 Colab 左側的「檔案」面板(資料夾圖示 📁),直接把兩個檔案拖進去:
#     test_for_q2.zip                 (17 MB)
#     checkpoint_best_regular (4).pth (123 MB)
# 拖曳可以一次多檔、也支援大檔,比 files.upload() 的小視窗穩定很多
#(那個視窗重複執行時常常不出現或卡住)。
#
# 上傳中左下角會顯示進度。**等進度跑完再執行本 cell。**
#
# 大檔若還是傳不上去 → 用最下面的 Google Drive 方式。
# ────────────────────────────────────────────────────────────────
import glob
import os
import zipfile

os.makedirs("/content/test", exist_ok=True)

# 找 zip 並解壓(不論放在 /content 或 /content/test)
for z in glob.glob("/content/*.zip") + glob.glob("/content/test/*.zip"):
    with zipfile.ZipFile(z) as f:
        f.extractall("/content/test")
    print(f"已解壓 {os.path.basename(z)}")

ann = glob.glob("/content/test/**/_annotations.coco.json", recursive=True)
pth = sorted(glob.glob("/content/**/*.pth", recursive=True))
ANN_PATH = ann[0] if ann else None
WEIGHTS = pth[0] if pth else None

print()
print("=" * 60)
if ANN_PATH:
    n = len(glob.glob(os.path.join(os.path.dirname(ANN_PATH), "*.jpg")))
    print(f"✅ 測試集  {n} 張影像")
else:
    print("✗ 測試集   還沒看到 —— 請把 test_for_q2.zip 拖進左側檔案面板")
print(f"{'✅ 微調權重' if WEIGHTS else '✗ 微調權重'}  {WEIGHTS or '還沒看到 —— 請把 .pth 拖進左側檔案面板'}")
print("=" * 60)

if ANN_PATH and WEIGHTS:
    print("兩個都齊了,往下跑 cell 3")
else:
    print("補上缺的檔案後,再執行一次本 cell(它只是掃描,重跑很安全)")

# ── 替代方案:大檔傳不上去就用 Google Drive ──────────────────────
# 把權重丟到自己的雲端硬碟,取消下面三行的註解並改成你的路徑:
#
# from google.colab import drive
# drive.mount('/content/drive')
# WEIGHTS = '/content/drive/MyDrive/checkpoint_best_regular (4).pth'


In [ ]:
# 3) 縮放 + 評估工具
#    shrink_pad 與 scripts/eval_smallobject.py 同一套方法:整張圖縮 s 倍貼回
#    原尺寸畫布(灰底),讓「物件佔畫面比例」真的變小 —— 只縮圖是沒用的,
#    因為模型會把輸入 resize 回固定尺寸,佔比不變。
import glob, json, os
import numpy as np
from PIL import Image

PAD = (114, 114, 114)
ANN = ANN_PATH          # 由 cell 2 決定,避免兩處各自 glob 不一致
IMG_DIR = os.path.dirname(ANN)
PERSON_CAT = 1          # COCO json 的「人」;模型輸出 class_id 為 0(差 1)
PERSON_CLS = 0


def shrink_pad(pil, s):
    """把圖縮 s 倍貼到原尺寸畫布(灰底)。回傳 (canvas, ox, oy)。"""
    W, H = pil.size
    nw, nh = max(int(W * s), 1), max(int(H * s), 1)
    canvas = Image.new("RGB", (W, H), PAD)
    ox, oy = (W - nw) // 2, (H - nh) // 2
    canvas.paste(pil.resize((nw, nh)), (ox, oy))
    return canvas, ox, oy


def load_person_gt():
    d = json.load(open(ANN, encoding="utf-8"))
    id2fn = {im["id"]: im["file_name"] for im in d["images"]}
    by = {}
    for a in d["annotations"]:
        if a["category_id"] != PERSON_CAT:
            continue
        x, y, w, h = a["bbox"]
        by.setdefault(id2fn[a["image_id"]], []).append([x, y, x + w, y + h])
    return by


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0


GT = load_person_gt()
print(f"test 影像 {len(GT)} 張、人框 {sum(len(v) for v in GT.values())} 個")
print("原始人框高度中位數 %.0f px" % np.median(
    [b[3] - b[1] for v in GT.values() for b in v]))

In [ ]:
# 4) 掃描:縮放 s × 輸入解析度
from rfdetr import RFDETRNano

SCALES = [1.0, 0.7, 0.5, 0.35, 0.25]
RESOLUTIONS = [704, 1280]
THR = 0.3        # 與 M3 評估一致

rows = []
for R in RESOLUTIONS:
    model = RFDETRNano(pretrain_weights=WEIGHTS, num_classes=11, resolution=R)
    for s in SCALES:
        tp = fn = fp = 0
        heights = []
        for fname, boxes in GT.items():
            img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
            canvas, ox, oy = shrink_pad(img, s)
            gt = [[b[0]*s+ox, b[1]*s+oy, b[2]*s+ox, b[3]*s+oy] for b in boxes]
            heights += [g[3] - g[1] for g in gt]
            det = model.predict(canvas, threshold=THR)
            pred = [det.xyxy[i].tolist() for i in range(len(det))
                    if int(det.class_id[i]) == PERSON_CLS] if len(det) else []
            used = [False] * len(gt)
            for p in pred:
                best, bi = 0.5, -1
                for j, g in enumerate(gt):
                    if used[j]:
                        continue
                    v = iou(p, g)
                    if v >= best:
                        best, bi = v, j
                if bi >= 0:
                    used[bi] = True
                    tp += 1
                else:
                    fp += 1
            fn += used.count(False)
        rec = tp / max(tp + fn, 1)
        prec = tp / max(tp + fp, 1)
        rows.append(dict(resolution=R, scale=s, recall=round(rec, 4),
                         precision=round(prec, 4),
                         person_px=round(float(np.median(heights)), 1),
                         tp=tp, fn=fn, fp=fp))
        print(f"  R={R} s={s:.2f}  人框中位高 {rows[-1]['person_px']:>5.1f}px  "
              f"Recall {rec:.3f}  Precision {prec:.3f}")
    model = None

In [ ]:
# 5) 結果表 + 判定
def verdict(r):
    return "可用" if r >= 0.90 else ("勉強" if r >= 0.75 else "不可用")

print("=" * 72)
print("人的偵測 vs 畫面中的表觀大小")
print("=" * 72)
print(f"{'解析度':>6}{'縮放':>7}{'人框高(px)':>12}{'Recall':>9}{'Precision':>11}  判定")
print("-" * 72)
for r in rows:
    print(f"{r['resolution']:>6}{r['scale']:>7.2f}{r['person_px']:>12.1f}"
          f"{r['recall']:>9.3f}{r['precision']:>11.3f}  {verdict(r['recall'])}")

print()
print("換算成部署參數(s 越小 = 視野越廣 = 同一台鏡頭涵蓋越大範圍):")
print("  s=0.50 → 視野寬度約 2 倍、涵蓋面積約 4 倍")
print("  s=0.35 → 視野寬度約 2.9 倍、涵蓋面積約 8 倍")
print("  s=0.25 → 視野寬度約 4 倍、涵蓋面積約 16 倍")
print()
for R in sorted({r["resolution"] for r in rows}):
    ok = [r for r in rows if r["resolution"] == R and r["recall"] >= 0.90]
    if ok:
        w = min(r["scale"] for r in ok)
        px = [r["person_px"] for r in ok if r["scale"] == w][0]
        print(f"  解析度 {R}:最廣可到 s={w:.2f}(人框高 {px:.0f}px)仍 Recall≥0.90")
    else:
        print(f"  解析度 {R}:**沒有任何縮放達到 Recall≥0.90**")

json.dump(rows, open("/content/q2_person_scale.json", "w"), indent=2)
from google.colab import files
files.download("/content/q2_person_scale.json")

## 怎麼判讀

**主判準是 Recall**(預先登記 §4 已定死),不是 mAP —— M5 只需要「人被偵測到」,
框準不準由 M4 的 Kalman 濾波吸收。

**最關鍵的那一格**:找出 Recall 仍 ≥ 0.90 的**最小 s**。那決定了一台鏡頭
最多能涵蓋多大範圍:

- 若 s=0.25 仍可用 → 一台鏡頭涵蓋 16 倍面積,**全景方案可行**
- 若只到 s=0.5 → 涵蓋 4 倍面積,可能需要**兩台半景鏡頭**而不是一台全景
- 若 s=0.7 就掉到 0.75 以下 → **全景方案不可行**,要回頭考慮徽章/腕帶

**「人框高(px)」是最實用的欄位**:現場勘查時量一下「人在預定鏡頭位置的畫面裡
大概幾個像素高」,直接對照這一欄就知道行不行,不必換算角度與高度。

**若 Recall 落在 0.75~0.90**:把它當成 M4 斷軌率的增量,回頭餵進
`scripts/sim_m5_montecarlo.py` 的 `master_fragment_rate` 重跑,
看碎裂率會被推高到多少。

把結果表 + `q2_person_scale.json` 貼回來一起判讀。

## 相關
- 第五輪預先登記與 Q1 結果:`docs/M5_模擬預先登記_全景鏡頭_20260825.md`
- 既有的小物件實驗(刀,非人):`docs/M3_小物件模擬結果.md`、`M3_解析度回升結果.md`
- 同一套縮放方法:`scripts/eval_smallobject.py`